In [1]:
# ---------------------------------- LIBRARY IMPORTS AND FUNCTION DEFINITIONS ----------------------------------
import pandas as pd
from pathlib import Path
import duckdb
import sqlalchemy
import sys, os
from dotenv import load_dotenv
from urllib.parse import quote_plus

sys.path.append(os.path.abspath(".."))
from utils.utils import data_add_moer
import pyarrow as pa
import pyarrow.parquet as pq
import psycopg2
import shutil
import gc

sys.path.append(os.path.abspath(".."))
from config import CONFIG

load_dotenv()

inventory_year = CONFIG['latest_inventory_year']
ers_baseline_year = CONFIG['ers_baseline_year']

user = quote_plus(os.getenv("CLIMATETRACE_USER"))
password = quote_plus(os.getenv("CLIMATETRACE_PASS"))
host = os.getenv("CLIMATETRACE_HOST")
port = '5678'
database = os.getenv("CLIMATETRACE_DB")

postgres_url = f"postgresql://{user}:{password}@{host}:{port}/{database}"


def split_or_move_parquet(input_file, output_dir, target_size_mb=40):
    """
    Splits a large Parquet file into ~target_size_mb chunks or moves it directly 
    to desdired folder if under the threshold.
    
    Args:
        input_file (str): Path to input parquet file.
        output_dir (str): Directory to save chunked (or moved) parquet files.
        target_size_mb (int): Approx size limit per chunk in MB.
    """
    input_file = Path(input_file)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    file_size_mb = input_file.stat().st_size / (1024 * 1024)

    # If file is already below threshold, just move it
    if file_size_mb <= target_size_mb:
        dest_file = output_dir / input_file.name
        shutil.move(str(input_file), dest_file)
        print(f"✅ File {input_file.name} was {file_size_mb:.1f} MB, moved to {dest_file}")
        return

    print(f"⚡ Splitting {input_file.name} ({file_size_mb:.1f} MB)...")

    # Load full Parquet into DataFrame
    df = pd.read_parquet(input_file)
    total_rows = len(df)

    # Estimate bytes per row
    test_sample = df.iloc[:min(10000, total_rows)]
    test_table = pa.Table.from_pandas(test_sample)
    pq.write_table(test_table, "temp.parquet")
    bytes_per_row = os.path.getsize("temp.parquet") / len(test_sample)
    os.remove("temp.parquet")

    # Rows per chunk
    target_bytes = target_size_mb * 1024 * 1024
    rows_per_chunk = max(1, int(target_bytes / bytes_per_row))

    # Split and write
    for i, start in enumerate(range(0, total_rows, rows_per_chunk)):
        end = min(start + rows_per_chunk, total_rows)
        chunk_df = df.iloc[start:end]
        chunk_table = pa.Table.from_pandas(chunk_df)
        output_path = output_dir / f"{input_file.stem}_chunk_{i+1}.parquet"
        pq.write_table(chunk_table, output_path)
        size_mb = output_path.stat().st_size / (1024 * 1024)
        print(f"  - Saved {output_path} ({size_mb:.1f} MB, rows {start}–{end})")

    # Delete original after chunking
    input_file.unlink()
    print(f"🗑️ Deleted original {input_file.name}")
    print("✅ Splitting complete")



In [2]:
# ---------------------------------- ARCHIVE THE EXISTING DATA ----------------------------------


def archive_parquets(data_folder_name='data', archive_folder_name='zzz_archive'):

    """
    Recursively finds ALL .parquet files under /data (at any depth),
    excluding certain root-level folders, and moves them to zzz_archive.
    Existing parquets in zzz_archive are removed before copying.
    """

    data_dir = Path.cwd()
    archive_dir = data_dir / archive_folder_name

    # Root-level folders/files to ignore
    ignore_list = {
        'percentile_moer',
        'raw_csvs',
        'strategy',
        'zzz_archive',
        'zzz_landing_zone',
        'README.md',
        'refresh_data.ipynb'
    }

    # ----- DELETE existing parquets in archive -----
    print("Clearing zzz_archive...")
    for f in archive_dir.glob("*.parquet"):
        f.unlink()
        print(f"Deleted old parquet: {f.name}")

    # ----- RECURSIVELY FIND ALL PARQUETS -----
    print("\nScanning for parquet files...\n")

    # Walk EVERY file in /data recursively
    for parquet_file in data_dir.rglob("*.parquet"):

        # Skip files **inside** ignored root-level directories
        parts = parquet_file.relative_to(data_dir).parts

        # parts[0] is the top-level folder name
        if parts and parts[0] in ignore_list:
            # skip parquets inside ignored dirs
            continue

        # Skip archive folder (we just cleared it)
        if archive_folder_name in parquet_file.parts:
            continue

        # Destination in archive
        dest = archive_dir / parquet_file.name

        # Move + overwrite if needed (archive is already emptied)
        shutil.move(str(parquet_file), str(dest))
        print(f"Moved: {parquet_file} → {dest}")

    print("\n✨ Complete: All parquet files archived. ✨")


archive_parquets()


Clearing zzz_archive...
Deleted old parquet: asset_aggregated_chunk_7.parquet
Deleted old parquet: gadm_0_emissions.parquet
Deleted old parquet: asset_aggregated_chunk_6.parquet
Deleted old parquet: global_heatmap_emissions_country.parquet
Deleted old parquet: asset_ownership.parquet
Deleted old parquet: gadm_1_emissions.parquet
Deleted old parquet: country_subsector_emissions_statistics_202602.parquet
Deleted old parquet: global_heatmap_emissions_totals.parquet
Deleted old parquet: asset_aggregated_chunk_4.parquet
Deleted old parquet: gadm_1_emissions_statistics_202602.parquet
Deleted old parquet: country_region_mapping.parquet
Deleted old parquet: asset_aggregated_chunk_5.parquet
Deleted old parquet: asset_aggregated_chunk_12.parquet
Deleted old parquet: asset_aggregated_chunk_9.parquet
Deleted old parquet: gadm_2_emissions_chunk_1.parquet
Deleted old parquet: asset_aggregated_chunk_8.parquet
Deleted old parquet: city_emissions.parquet
Deleted old parquet: asset_emissions_country_sub

In [3]:
###### NEW CODE TO HANDLE UNKNOWN non-broadcasting-vessels


"""
This script takes CSVs dropped into data/zzz_landing_zone,
routes them based on filename, and converts them into Parquet.
If the new parquet is larger than 45MB, it will be split into
smaller chunks before routing to its destination folder.

⚠️ Only processes .csv files
"""

from pathlib import Path
import pandas as pd

input_dir = Path("zzz_landing_zone")
output_base = Path("statistics")

routing_map = {
    "country_subsector_emissions_statistics": "country_subsector_emissions_statistics",
    "country_subsector_emissions_totals": "country_subsector_emissions_totals",
    "gadm_1_emissions_statistics": "gadm_1_emissions_statistics",
}

# Filenames that require the one-off fix before parquet conversion
FIX_PATTERNS = {
    "country_subsector_emissions_statistics",
    "country_subsector_emissions_totals",
}

def apply_unknown_country_name_fix(df: pd.DataFrame) -> pd.DataFrame:
    """
    For the record where:
      - country_name is null
      - subsector == 'non-broadcasting-vessles'
      - iso3_country == 'UNK'
    set country_name = 'Unknown'

    Expected to update exactly one record.
    """
    required_cols = {"country_name", "subsector", "iso3_country"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns for fix: {sorted(missing)}")

    mask = (
        df["country_name"].isna()
        & (df["subsector"] == "non-broadcasting-vessels")
        & (df["iso3_country"] == "UNK")
        & (df["gas"] == "co2e_100yr")
    )

    n = int(mask.sum())
    if n == 1:
        df.loc[mask, "country_name"] = "Unknown"
        print("🛠️ Applied fix: set country_name='Unknown' for UNK/non-broadcasting-vessles (1 row).")
    elif n == 0:
        print("ℹ️ Fix not applied: no matching row found (expected 1).")
    else:
        # Still apply, but flag loudly
        df.loc[mask, "country_name"] = "Unknown"
        print(f"⚠️ Fix matched {n} rows (expected 1). Updated them anyway—please investigate.")
    return df

# Ensure output subfolders exist
for subfolder in routing_map.values():
    (output_base / subfolder).mkdir(parents=True, exist_ok=True)

# Process only CSVs
for csv_file in input_dir.glob("*.csv"):
    print(f"Converting {csv_file.name}...")

    # Convert CSV → Parquet in landing zone
    df = pd.read_csv(csv_file)

    # Apply fix ONLY for the two relevant file types
    if any(pattern in csv_file.name for pattern in FIX_PATTERNS):
        df = apply_unknown_country_name_fix(df)

    parquet_file = input_dir / csv_file.with_suffix(".parquet").name
    df.to_parquet(parquet_file, engine="pyarrow", index=False)
    print(f"✅ Converted {csv_file.name} → {parquet_file.name} in landing zone")

    # Delete original CSV
    csv_file.unlink()
    print(f"🗑️ Deleted original CSV: {csv_file.name}")

    # Route Parquet into correct stats subfolder (split if needed)
    destination = None
    for pattern, subfolder in routing_map.items():
        if pattern in parquet_file.name:
            destination = output_base / subfolder
            break

    if destination:
        split_or_move_parquet(parquet_file, destination)
    else:
        print(f"⚠️ No matching subfolder for {parquet_file.name}, skipping.")

print("🎉 CSV to Parquet + routing complete.")


Converting country_subsector_emissions_totals_202604.csv...


/var/folders/dz/vqfc7snj23qc01ndgmlznjtc0000gn/T/ipykernel_43489/3700922090.py:74: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_file)


⚠️ Fix matched 62 rows (expected 1). Updated them anyway—please investigate.
✅ Converted country_subsector_emissions_totals_202604.csv → country_subsector_emissions_totals_202604.parquet in landing zone
🗑️ Deleted original CSV: country_subsector_emissions_totals_202604.csv
✅ File country_subsector_emissions_totals_202604.parquet was 30.2 MB, moved to statistics/country_subsector_emissions_totals/country_subsector_emissions_totals_202604.parquet
Converting country_subsector_emissions_statistics_202604.csv...
🛠️ Applied fix: set country_name='Unknown' for UNK/non-broadcasting-vessles (1 row).
✅ Converted country_subsector_emissions_statistics_202604.csv → country_subsector_emissions_statistics_202604.parquet in landing zone
🗑️ Deleted original CSV: country_subsector_emissions_statistics_202604.csv
✅ File country_subsector_emissions_statistics_202604.parquet was 4.7 MB, moved to statistics/country_subsector_emissions_statistics/country_subsector_emissions_statistics_202604.parquet
Convert

In [ ]:
# from pathlib import Path
# import duckdb

# # Choose the folder you want to inspect
# # parquet_folder = Path("statistics/country_subsector_emissions_statistics")
# # or:
# parquet_folder = Path("statistics/country_subsector_emissions_totals")

# parquet_path = str(parquet_folder / "*.parquet")

# # Output CSV file
# output_csv = Path("unknown_country_records.csv")

# con = duckdb.connect()

# query = f"""
# COPY (
#     SELECT *
#     FROM read_parquet('{parquet_path}')
#     WHERE country_name = 'Unknown'
# ) TO '{output_csv}' (FORMAT CSV, HEADER TRUE);
# """

# con.execute(query)

# print(f"✅ Exported records with country_name='Unknown' to {output_csv}")


In [ ]:
# """
# This script takes CSVs dropped into data/zzz_landing_zone,
# routes them based on filename, and converts them into Parquet.
# If the new parquet is larger than 45MB, it will be split into 
# smaller chunks before routing to its destination folder.

# ⚠️ Only processes .csv files
# """

# input_dir = Path("zzz_landing_zone")
# output_base = Path("statistics")

# routing_map = {
#     "country_subsector_emissions_statistics": "country_subsector_emissions_statistics",
#     "country_subsector_emissions_totals": "country_subsector_emissions_totals",
#     "gadm_1_emissions_statistics": "gadm_1_emissions_statistics"
# }

# # Ensure output subfolders exist
# for subfolder in routing_map.values():
#     (output_base / subfolder).mkdir(parents=True, exist_ok=True)

# # Process only CSVs
# for csv_file in input_dir.glob("*.csv"):
#     print(f"Converting {csv_file.name}...")

#     # Convert CSV → Parquet in landing zone
#     df = pd.read_csv(csv_file)
#     parquet_file = input_dir / csv_file.with_suffix(".parquet").name
#     df.to_parquet(parquet_file, engine="pyarrow", index=False)
#     print(f"✅ Converted {csv_file.name} → {parquet_file.name} in landing zone")

#     # Delete original CSV
#     csv_file.unlink()
#     print(f"🗑️ Deleted original CSV: {csv_file.name}")

#     # Route Parquet into correct stats subfolder (split if needed)
#     destination = None
#     for pattern, subfolder in routing_map.items():
#         if pattern in parquet_file.name:
#             destination = output_base / subfolder
#             break

#     if destination:
#         split_or_move_parquet(parquet_file, destination)
#     else:
#         print(f"⚠️ No matching subfolder for {parquet_file.name}, skipping.")

# print("🎉 CSV to Parquet + routing complete.")

🎉 CSV to Parquet + routing complete.


In [6]:
# -------------------- Heatmap Global Inventory View ----------------------
#### this data is for heatmap only when Global Emissions Inventory is selected as inputs
#### it is here because of nuances in getting non-broadcasting-vessels included/aggregated correctly

from sqlalchemy import text

year = CONFIG['latest_inventory_year']

parquet_path_totals = "zzz_landing_zone/global_heatmap_emissions_totals.parquet"
parquet_path_country = "zzz_landing_zone/global_heatmap_emissions_country.parquet"

output_path = "heatmap_global_inventory_view/"

engine = sqlalchemy.create_engine(postgres_url)

query_totals = f'''
    select cast('Total' as text) as Region
        , sum(case when sector = 'agriculture' then emissions_quantity else 0 end) as agriculture
        , sum(case when sector = 'buildings' then emissions_quantity else 0 end) as buildings
        , sum(case when sector = 'fluorinated-gases' then emissions_quantity else 0 end) as fluorinated_gases
        , sum(case when sector = 'fossil-fuel-operations' then emissions_quantity else 0 end) as fossil_fuel_operations
        , sum(case when sector = 'manufacturing' then emissions_quantity else 0 end) as manufacturing
        , sum(case when sector = 'mineral-extraction' then emissions_quantity else 0 end) as mineral_extraction
        , sum(case when sector = 'power' then emissions_quantity else 0 end) as power
        , sum(case when sector = 'transportation' then emissions_quantity else 0 end) as transportation
        , sum(case when sector = 'waste' then emissions_quantity else 0 end) as waste
        , sum(case when sector <> 'forestry-and-land-use' then emissions_quantity else 0 end) as total_exc_forestry
        , sum(case when sector = 'forestry-and-land-use' then emissions_quantity else 0 end) forestry_and_land_use
        , sum(emissions_quantity) as total_emissions_quantity
        --, count(distinct asset_id) asset_count 

    from public.country_emissions cem
    left join (
        select distinct sector, subsector from public.asset_schema
    ) asch
        on cast(asch.subsector as text) = cast(cem.original_inventory_sector as text)

    where gas = 'co2e_100yr'
        and extract(year from start_time) = {year}
'''

print('Query Running: Aggregating global emissions totals for heatmap...')
df_totals = pd.read_sql_query(text(query_totals), engine)
df_totals.to_parquet(parquet_path_totals, index=False)
split_or_move_parquet(parquet_path_totals, output_path)

query_country = f'''
    select ca.name as Region
        , sum(case when sector = 'agriculture' then emissions_quantity else 0 end) as agriculture
        , sum(case when sector = 'buildings' then emissions_quantity else 0 end) as buildings
        , sum(case when sector = 'fluorinated-gases' then emissions_quantity else 0 end) as fluorinated_gases
        , sum(case when sector = 'fossil-fuel-operations' then emissions_quantity else 0 end) as fossil_fuel_operations
        , sum(case when sector = 'manufacturing' then emissions_quantity else 0 end) as manufacturing
        , sum(case when sector = 'mineral-extraction' then emissions_quantity else 0 end) as mineral_extraction
        , sum(case when sector = 'power' then emissions_quantity else 0 end) as power
        , sum(case when sector = 'transportation' then emissions_quantity else 0 end) as transportation
        , sum(case when sector = 'waste' then emissions_quantity else 0 end) as waste
        , sum(case when sector <> 'forestry-and-land-use' then emissions_quantity else 0 end) as total_exc_forestry
        , sum(case when sector = 'forestry-and-land-use' then emissions_quantity else 0 end) forestry_and_land_use
        , sum(emissions_quantity) as total_emissions_quantity
        --, count(distinct asset_id) asset_count 

    from public.country_emissions cem
    left join (
        select distinct sector, subsector from public.asset_schema
    ) asch
        on cast(asch.subsector as text) = cast(cem.original_inventory_sector as text)
    left join public.country_analysis ca
        on cast(ca.iso3_country as text) = cast(cem.iso3_country as text)

    where gas = 'co2e_100yr'
        and extract(year from start_time) = {year}

    group by ca.name
'''

print('Query Running: Aggregating country-sector emissions for heatmap...')
df_country = pd.read_sql_query(text(query_country), engine)
df_country.to_parquet(parquet_path_country, index=False)
split_or_move_parquet(parquet_path_country, output_path)

engine.dispose()

Query Running: Aggregating global emissions totals for heatmap...
✅ File global_heatmap_emissions_totals.parquet was 0.0 MB, moved to heatmap_global_inventory_view/global_heatmap_emissions_totals.parquet
Query Running: Aggregating country-sector emissions for heatmap...
✅ File global_heatmap_emissions_country.parquet was 0.0 MB, moved to heatmap_global_inventory_view/global_heatmap_emissions_country.parquet


In [ ]:
# 1. ------------ THIS NEEDS TO BE FIXED AND PROBABLY BE STORED AT ASSET LEVEL... TRY CALCULATING AT ASSET LEVEL FIRST
# 2. CHECK OTHER TABLES FOR TEMP GRAIN (GADM & CITY)

# ------------------------------------- ASSET EMISSIONS COUNTRY SUBSECTOR LEVEL -------------------------------------

parquet_path = "zzz_landing_zone/asset_emissions_country_subsector.parquet"
output_path = "asset_emissions/monthly_country_subsector_level"

# Use DuckDB to write directly from PostgreSQL to Parquet
con = duckdb.connect()


print("Getting max month...")
max_date = con.execute(f"""
    select max(start_time)
    from postgres_scan('{postgres_url}', 'public', 'asset_emissions')                       
""").fetchone()[0]

print("Aggregating assets to subsector-level and writing to parquet file, this may take a while...")
con.execute(f"""
    INSTALL postgres;
    LOAD postgres;

    CREATE TABLE asset_emissions_parquet AS
    SELECT ae.iso3_country,
        ae.original_inventory_sector,
        itm.activity_is_temporal,
        ae.start_time,
        ae.gas,
        sch.sector,
        ca.name as country_name,
        ca.continent,
        ca.unfccc_annex,
        ca.em_finance,
        ca.g20,
        ca.eu,
        ca.oecd,
        ca.developed_un,
        ae.release,
        ae.activity_units,
        sum(emissions_quantity) emissions_quantity,
        case when activity_is_temporal = true then sum(activity) else avg(activity) end as activity,
        sum(emissions_quantity) / sum(activity) weighted_average_emissions_factor
    
    FROM postgres_scan('{postgres_url}', 'public', 'asset_emissions') ae
    LEFT JOIN postgres_scan('{postgres_url}', 'public', 'country_analysis') ca
        ON CAST(ca.iso3_country AS VARCHAR) = CAST(ae.iso3_country AS VARCHAR)
    LEFT JOIN (
        SELECT DISTINCT sector, subsector FROM postgres_scan('{postgres_url}', 'public', 'asset_schema')
    ) sch
        ON CAST(sch.subsector AS VARCHAR) = CAST(ae.original_inventory_sector AS VARCHAR)
    left join postgres_scan('{postgres_url}', 'public', 'is_temporal_map') itm
        on itm.original_inventory_sector = ae.original_inventory_sector
    
    WHERE ae.start_time >= (
                date_trunc('year', DATE '{max_date}') - INTERVAL '3 YEARS'
            )
      AND ae.gas in ('co2e_100yr','ch4')
      AND ae.most_granular = TRUE
    
    GROUP BY ae.iso3_country,
        ae.original_inventory_sector,
        itm.activity_is_temporal,
        ae.start_time,
        ae.gas,
        sch.sector,
        ca.name,
        ca.continent,
        ca.unfccc_annex,
        ca.em_finance,
        ca.g20,
        ca.eu,
        ca.oecd,
        ca.developed_un,
        ae.release,
        ae.activity_units;

    COPY asset_emissions_parquet TO '{parquet_path}' (FORMAT PARQUET);
""")
con.close()

split_or_move_parquet(parquet_path, output_path)

print("✅ Asset parquet file exported")

Getting max month...


In [ ]:
# ------------------------------------ Asset Annual Emissions ------------------------------------

################### CURRENTLY USING DATA FUSION TABLES, NEEDS TO BE CHANGED BACK WHEN READY
from sqlalchemy import text

parquet_path = "zzz_landing_zone/asset_annual_emissions.parquet"

engine = sqlalchemy.create_engine(postgres_url)


print('Query Running: Aggregating asset data to annual level and adding ERS...')
query = f'''

		select extract(year from ae.start_time) as year
			, cast(ae.asset_id as text) as asset_id
			, ai.asset_type
			, CASE 
					WHEN ae.original_inventory_sector = 'iron-and-steel' AND ai.asset_type LIKE '%BF%' 
						THEN '{{''iron-and-steel'': [''BF'', ''DRI-EAF'']}}'
					WHEN ae.original_inventory_sector = 'aluminum' AND ai.asset_type LIKE '%Refinery%' 
						THEN '{{''aluminum'': [''Refinery'']}}'
					WHEN ae.original_inventory_sector = 'aluminum' AND ai.asset_type LIKE '%Smelting%' 
						THEN '{{''aluminum'': [''Smelting'']}}'
					ELSE 'all' 
				END AS asset_type_2
			, ai.asset_name
			, ae.iso3_country
			, ca.name as country_name
			, abc.region balancing_authority_region
			, ca.continent
			, ca.eu
			, ca.oecd
			, ca.unfccc_annex
			, ca.developed_un
			, ca.em_finance
            , ca.g20
			, asch.sector
			, ae.original_inventory_sector as subsector
            , itm.capacity_is_temporal
			, itm.activity_is_temporal
			, ST_AsText(al.location) as lat_lon
			, al.gadm_1
			, al.gadm_2
            , ae.most_granular
			, al.ghs_fua
			, al.city_id
			, ae.other1
			, ae.other2
			, ae.other3
			, ae.other4
			, ae.other5
			, ae.other6
			, ae.other7
			, ae.other8
			, ae.other9
			, ae.other10
			, ae.activity_units
			, case when capacity_is_temporal = true then sum(capacity) else avg(capacity) end as capacity
			, case when activity_is_temporal = true then sum(activity) else avg(activity) end as activity
			, avg(emissions_factor) average_emissions_factor
			, sum(emissions_quantity) emissions_quantity
			, 'asset' as reduction_q_type
			, ers.strategy_id
			, ers.strategy_name
			, ers.strategy_description
			, ers.mechanism
			, ers.old_activity
			, ers.affected_activity
			, ers.old_emissions_factor
			, ers.new_emissions_factor
			, ers.emissions_reduced_at_asset
			, ers.induced_sector_1
			, ers.induced_sector_1_induced_emissions
			, ers.induced_sector_2
			, ers.induced_sector_2_induced_emissions
			, ers.induced_sector_3
			, ers.induced_sector_3_induced_emissions
			, ers.total_emissions_reduced_per_year
            , ers.feasibility
            , ers.feasibility_score
            , ers.cost
            , ers.cost_score
            , ers.asset_rf_score
            , ers.asset_difficulty_score

		from public.asset_emissions_data_fusion ae
		left join public.asset_information_data_fusion ai
			on ai.asset_id = ae.asset_id
		left join public.asset_location_data_fusion al
			on al.asset_id = ae.asset_id
		left join (
			select distinct sector, subsector from public.asset_schema
		) asch
			on cast(asch.subsector as varchar) = cast(ae.original_inventory_sector as varchar)
		left join public.country_analysis ca
			on cast(ca.iso3_country as varchar) = cast(ae.iso3_country as varchar)
		left join public.asset_ba_crosswalk abc
			on cast(abc.asset_id as text) = cast(ae.asset_id as text)
		left join (
			select rdf.* 
			from public.reductions_data_fusion rdf
			where strategy_rank = 1
				and rdf.gas = 'co2e_100yr'
		) ers
			on ers.asset_id = ae.asset_id
		left join public.is_temporal_map itm
			on cast(itm.original_inventory_sector as text) = cast(ae.original_inventory_sector as text)

		where extract(year from ae.start_time) = {ers_baseline_year}
			and ae.gas = 'co2e_100yr'

		group by extract(year from ae.start_time)
			, ae.asset_id
			, ai.asset_type
			, CASE 
					WHEN ae.original_inventory_sector = 'iron-and-steel' AND ai.asset_type LIKE '%BF%' 
						THEN '{{''iron-and-steel'': [''BF'', ''DRI-EAF'']}}'
					WHEN ae.original_inventory_sector = 'aluminum' AND ai.asset_type LIKE '%Refinery%' 
						THEN '{{''aluminum'': [''Refinery'']}}'
					WHEN ae.original_inventory_sector = 'aluminum' AND ai.asset_type LIKE '%Smelting%' 
						THEN '{{''aluminum'': [''Smelting'']}}'
					ELSE 'all' 
				END
			, ai.asset_name
			, ae.iso3_country
			, ca.name
			, abc.region
			, ca.continent
			, ca.eu
			, ca.oecd
			, ca.unfccc_annex
			, ca.developed_un
			, ca.em_finance
            , ca.g20
			, asch.sector
			, ae.original_inventory_sector
            , itm.capacity_is_temporal
			, itm.activity_is_temporal
			, ST_AsText(al.location)
			, al.gadm_1
			, al.gadm_2
            , ae.most_granular
			, al.ghs_fua
			, al.city_id
			, ae.other1
			, ae.other2
			, ae.other3
			, ae.other4
			, ae.other5
			, ae.other6
			, ae.other7
			, ae.other8
			, ae.other9
			, ae.other10
			, ae.activity_units
			, ers.strategy_id
			, ers.strategy_name
			, ers.strategy_description
			, ers.mechanism
			, ers.old_activity
			, ers.affected_activity
			, ers.old_emissions_factor
			, ers.new_emissions_factor
			, ers.emissions_reduced_at_asset
			, ers.induced_sector_1
			, ers.induced_sector_1_induced_emissions
			, ers.induced_sector_2
			, ers.induced_sector_2_induced_emissions
			, ers.induced_sector_3
			, ers.induced_sector_3_induced_emissions
			, ers.total_emissions_reduced_per_year
            , ers.feasibility
            , ers.feasibility_score
            , ers.cost
            , ers.cost_score
            , ers.asset_rf_score
            , ers.asset_difficulty_score
			
			UNION ALL
			
			SELECT 
				{ers_baseline_year} AS year,
				asset_id,
				gr.asset_type,
				NULL AS asset_type_2,
				gr.asset_name,
				ca.iso3_country,
				ca.name AS country_name,
				NULL AS balancing_authority_region,
				ca.continent,
				ca.eu,
				ca.oecd,
				ca.unfccc_annex,
				ca.developed_un,
				ca.em_finance,
                ca.g20,
				asch.sector,
				gr.original_inventory_sector AS subsector,
                itm.capacity_is_temporal,
				itm.activity_is_temporal,
				null as lat_lon,
				CASE 
					WHEN gb.admin_level = 1 THEN gb.gadm_id
					WHEN gb.admin_level = 2 THEN gb.immediate_parent 
					ELSE NULL 
				END AS gadm_1,
				CASE 
					WHEN gb.admin_level = 2 THEN gb.gadm_id 
					ELSE NULL 
				END AS gadm_2,
                true AS most_granular,
				NULL AS ghs_fua,
				NULL AS city_id,
				NULL AS other1,
				NULL AS other2,
				NULL AS other3,
				NULL AS other4,
				NULL AS other5,
				NULL AS other6,
				NULL AS other7,
				NULL AS other8,
				NULL AS other9,
				NULL AS other10,
				NULL AS activity_units,
				0 AS capacity,
				0 AS activity,
				0 AS average_emissions_factor,
				gr.baseline_emissions AS emissions_quantity,
				'remainder' AS reduction_q_type,
				gr.strategy_id,
				gr.strategy_name,
				gr.strategy_description,
				gr.mechanism,
				gr.old_activity,
				gr.affected_activity,
				gr.old_emissions_factor,
				gr.new_emissions_factor,
				gr.emissions_reduced_at_asset,
				gr.induced_sector_1,
				gr.induced_sector_1_induced_emissions,
				gr.induced_sector_2,
				gr.induced_sector_2_induced_emissions,
				gr.induced_sector_3,
				gr.induced_sector_3_induced_emissions,
				gr.total_emissions_reduced_per_year,
                null as feasibility,
				null as feasibility_score,
				null as cost,
				null as cost_score,
				null as asset_rf_score,
				null as asset_difficulty_score
			
			FROM public.gadm_reductions_data_fusion gr
			LEFT JOIN (
				select distinct sector, subsector from public.asset_schema
			) asch
				on cast(asch.subsector as varchar) = cast(gr.original_inventory_sector as varchar)
			LEFT JOIN (
				select distinct gadm_id, iso3_country, admin_level, immediate_parent
				from public.gadm_boundaries
			) gb
				on gb.gadm_id = gr.asset_id
			LEFT JOIN public.country_analysis ca
				on ca.iso3_country = gb.iso3_country
			LEFT JOIN public.is_temporal_map itm
				on cast(itm.original_inventory_sector as text) = cast(gr.original_inventory_sector as text)
				
			WHERE gr.strategy_rank = 1
				and gr.gas = 'co2e_100yr'
                and total_emissions_reduced_per_year > 0
                
            UNION ALL

            SELECT 
				{ers_baseline_year} AS year,
				asset_id,
				cr.asset_type,
				NULL AS asset_type_2,
				cr.asset_name,
				ca.iso3_country,
				ca.name AS country_name,
				NULL AS balancing_authority_region,
				ca.continent,
				ca.eu,
				ca.oecd,
				ca.unfccc_annex,
				ca.developed_un,
				ca.em_finance,
                ca.g20,
				asch.sector,
				cr.original_inventory_sector AS subsector,
                itm.capacity_is_temporal,
				itm.activity_is_temporal,
				null as lat_lon,
				null AS gadm_1,
				NULL AS gadm_2,
                false AS most_granular,
				array[cr.asset_id] AS ghs_fua,
				cr.asset_id AS city_id,
				NULL AS other1,
				NULL AS other2,
				NULL AS other3,
				NULL AS other4,
				NULL AS other5,
				NULL AS other6,
				NULL AS other7,
				NULL AS other8,
				NULL AS other9,
				NULL AS other10,
				NULL AS activity_units,
				0 AS capacity,
				0 AS activity,
				0 AS average_emissions_factor,
                cr.baseline_emissions as emissions_quantity,
				'remainder' AS reduction_q_type,
				cr.strategy_id,
				cr.strategy_name,
				cr.strategy_description,
				cr.mechanism,
				cr.old_activity,
				cr.affected_activity,
				cr.old_emissions_factor,
				cr.new_emissions_factor,
				cr.emissions_reduced_at_asset,
				cr.induced_sector_1,
				cr.induced_sector_1_induced_emissions,
				cr.induced_sector_2,
				cr.induced_sector_2_induced_emissions,
				cr.induced_sector_3,
				cr.induced_sector_3_induced_emissions,
				cr.total_emissions_reduced_per_year,
                null as feasibility,
				null as feasibility_score,
				null as cost,
				null as cost_score,
				null as asset_rf_score,
				null as asset_difficulty_score
			
			FROM public.city_reductions_data_fusion cr
			LEFT JOIN (
				select distinct sector, subsector from public.asset_schema
			) asch
				on cast(asch.subsector as varchar) = cast(cr.original_inventory_sector as varchar)

            ------ CHANGE THIS JOIN AHHHHHHH ------    
			LEFT JOIN (
				select distinct city_id, iso3_country
				from public.city_boundaries
			) cb
				on cb.city_id = cr.asset_id
                
            
			LEFT JOIN public.country_analysis ca
				on ca.iso3_country = cb.iso3_country
			LEFT JOIN public.is_temporal_map itm
				on cast(itm.original_inventory_sector as text) = cast(cr.original_inventory_sector as text)
				
			WHERE cr.strategy_rank = 1
				and cr.gas = 'co2e_100yr'
                and total_emissions_reduced_per_year > 0
                and asset_id not like '%_EXT'
        ;     
    '''

# print(query)

df = pd.read_sql_query(text(query), engine)

df.to_parquet(parquet_path, index=False)
# removing forestry sectors from query
		# and ae.original_inventory_sector not in ('forest-land-clearing',
		# 											'forest-land-degradation',
		# 											'forest-land-fires',
		# 											'net-forest-land',
		# 											'net-shrubgrass',
		# 											'net-wetland',
		# 											'removals',
		# 											'shrubgrass-fires',
		# 											'water-reservoirs',
		# 											'wetland-fires')

#  con.close()

## ---------------------------------- ADD MOER FACTORS --------------------------------------
parquet_path = Path('zzz_landing_zone/asset_annual_emissions.parquet')
landing_zone_path = Path('zzz_landing_zone/asset_annual_emissions_moer.parquet')
# output_path =  Path('asset_emissions/asset_level_2024')

df_asset = pd.read_parquet(parquet_path)

# adding moer data to assets
asset_moer_df = data_add_moer(df_asset, cond={"moer": True})

# converting data to new parquet file
asset_moer_df.to_parquet(landing_zone_path, index=False)

# freeing up memory
print("Deleting asset-moer dataframes to free up memory.")
del df_asset
del asset_moer_df
gc.collect()

# # deleting original asset file
parquet_path.unlink()
print("Original asset file deleted")

# print("Aggregating Asset Data")

engine.dispose()


# # splitting asset data into chunks
# split_or_move_parquet(landing_zone_path, output_path)

# print("Successfully updated asset_level data.")

In [ ]:
moer_path = Path('zzz_landing_zone/asset_annual_emissions_moer.parquet')
agg_path = Path('zzz_landing_zone/asset_aggregated.parquet')
output_path = Path('asset_emissions/annual_asset')

con = duckdb.connect()

con.execute(f'''
            create table asset_aggregated as
            select year
                , asset_id
                , asset_type
                , asset_type_2
                , asset_name
                , iso3_country
                , country_name
                , balancing_authority_region
                , continent
                , eu
                , oecd
                , unfccc_annex
                , developed_un
                , em_finance
                , g20
                , sector
                , subsector
                , capacity_is_temporal
                , activity_is_temporal
                , lat_lon
                , gadm_1
                , gadm_2
                , most_granular
                , ghs_fua
                , city_id
                , activity_units
                , case when capacity_is_temporal = true then sum(capacity) else avg(capacity) end as capacity
                , case when activity_is_temporal = true then sum(activity) else avg(activity) end as activity
                , avg(average_emissions_factor) average_emissions_factor
                , sum(emissions_quantity) emissions_quantity
                , reduction_q_type
                , strategy_id
                , strategy_name
                , strategy_description
                , mechanism
                , old_activity
                , affected_activity
                , old_emissions_factor
                , new_emissions_factor
                , emissions_reduced_at_asset
                , induced_sector_1
                , induced_sector_1_induced_emissions
                , induced_sector_2
                , induced_sector_2_induced_emissions
                , induced_sector_3
                , induced_sector_3_induced_emissions
                , total_emissions_reduced_per_year
                , feasibility
                , feasibility_score
                , cost
                , cost_score
                , asset_rf_score
                , asset_difficulty_score
                , avg(ef_moer) ef_moer
                , sum(eq_12) eq_12
                , avg(ef_12) ef_12
                , sum(eq_12_moer) eq_12_moer
                , avg(ef_12_moer) ef_12_moer
                
            from '{moer_path}'

            group by year
                , asset_id
                , asset_type
                , asset_type_2
                , asset_name
                , iso3_country
                , country_name
                , balancing_authority_region
                , continent
                , eu
                , oecd
                , unfccc_annex
                , developed_un
                , em_finance
                , g20
                , sector
                , subsector
                , capacity_is_temporal
                , activity_is_temporal
                , lat_lon
                , gadm_1
                , gadm_2
                , most_granular
                , ghs_fua
                , city_id
                , activity_units
                , reduction_q_type
                , strategy_id
                , strategy_name
                , strategy_description
                , mechanism
                , old_activity
                , affected_activity
                , old_emissions_factor
                , new_emissions_factor
                , emissions_reduced_at_asset
                , induced_sector_1
                , induced_sector_1_induced_emissions
                , induced_sector_2
                , induced_sector_2_induced_emissions
                , induced_sector_3
                , induced_sector_3_induced_emissions
                , total_emissions_reduced_per_year
                , feasibility
                , feasibility_score
                , cost
                , cost_score
                , asset_rf_score
                , asset_difficulty_score
            ;

            COPY asset_aggregated to '{agg_path}' (FORMAT PARQUET);
            
            ''')

con.close()

split_or_move_parquet(agg_path, output_path)

moer_path.unlink()


In [ ]:
# # -------------------------------------- Uncomment and run this if you created the file in the block above, but couldn't get it to split and moved to the correct folder

# parquet_path = Path('zzz_landing_zone/asset_annual_emissions.parquet')
# landing_zone_path = Path('zzz_landing_zone/asset_annual_emissions_moer.parquet')
# output_path =  Path('asset_emissions/asset_level_2024')

# df_asset = pd.read_parquet(parquet_path)

# # adding moer data to assets
# asset_moer_df = data_add_moer(df_asset, cond={"moer": True})

# # converting data to new parquet file
# asset_moer_df.to_parquet(landing_zone_path, index=False)

# freeing up memory
# print("Deleting asset-moer dataframes to free up memory.")
# del df_asset
# del asset_moer_df
# gc.collect()

# deleting original asset file
# parquet_path.unlink()
# print("Original asset file deleted")

# # splitting asset data into chunks
# split_or_move_parquet(landing_zone_path, output_path)

# print("Successfully updated asset_level data.")

In [ ]:
# ------------------------------------ GADM_0 Emissions ------------------------------------


parquet_path = "zzz_landing_zone/gadm_0_emissions.parquet"
output_path = "gadm_emissions/gadm_0"

# Use DuckDB to write directly from PostgreSQL to Parquet
con = duckdb.connect()


print('Running query')
con.execute(f'''
    INSTALL postgres;
    LOAD postgres;

    CREATE TABLE gadm_0_emissions_parquet AS
    select extract(year from g0e.start_time) as year 
        , g0e.gadm_id
        , gb.gid
        , gb.admin_level
        , g0e.iso3_country
        , ca.name as country_name
        , gb.name gadm_0_name
        , gb.corrected_name gadm_0_corrected_name
        , ca.continent
        , ca.eu
        , ca.oecd
        , ca.unfccc_annex
        , ca.developed_un
        , ca.em_finance
        , ca.g20
        , asch.sector
        , g0e.original_inventory_sector subsector
        , itm.activity_is_temporal
        , g0e.gas
        , case when activity_is_temporal = true then sum(asset_activity) else avg(asset_activity) end as asset_activity
        , sum(asset_emissions) asset_emissions
        , case when activity_is_temporal = true then sum(remainder_activity) else avg(remainder_activity) end as remainder_activity
        , sum(remainder_emissions) remainder_emissions
        , sum(asset_emissions) + sum(remainder_emissions) as emissions_quantity

    from postgres_scan('{postgres_url}', 'public', 'gadm_0_emissions') g0e
    left join (
        select distinct gadm_id
            , gid
            , name
            , corrected_name
            , admin_level
        from postgres_scan('{postgres_url}','public', 'gadm_boundaries') 
        where admin_level = 0
    ) as gb
        on g0e.gadm_id = gb.gadm_id
    left join (
        select distinct sector
            , subsector
        from postgres_scan('{postgres_url}','public', 'asset_schema') 
    ) asch
        on cast(asch.subsector as varchar) = cast(g0e.original_inventory_sector as varchar)
    left join postgres_scan('{postgres_url}','public', 'country_analysis') ca
		on cast(ca.iso3_country as varchar) = cast(g0e.iso3_country as varchar)
    left join postgres_scan('{postgres_url}', 'public', 'is_temporal_map') itm
         on itm.original_inventory_sector = g0e.original_inventory_sector

    where g0e.gas = 'co2e_100yr'
        and extract(year from start_time) = {ers_baseline_year}
        
    group by extract(year from g0e.start_time) 
        , g0e.gadm_id
        , gb.gid
        , gb.admin_level
        , g0e.iso3_country
        , ca.name
        , gb.name 
        , gb.corrected_name
        , ca.continent
        , ca.eu
        , ca.oecd
        , ca.unfccc_annex
        , ca.developed_un
        , ca.em_finance
        , ca.g20
        , asch.sector
        , g0e.original_inventory_sector
        , itm.activity_is_temporal
        , g0e.gas;

    COPY gadm_0_emissions_parquet TO '{parquet_path}' (FORMAT PARQUET);
''')

# and g0e.original_inventory_sector not in ('forest-land-clearing',
#                                                 'forest-land-degradation',
#                                                 'forest-land-fires',
#                                                 'net-forest-land',
#                                                 'net-shrubgrass',
#                                                 'net-wetland',
#                                                 'removals',
#                                                 'shrubgrass-fires',
#                                                 'water-reservoirs',
#                                                 'wetland-fires')


con.close()

split_or_move_parquet(parquet_path, output_path)

print('Successfully refreshed GADM_0 data.')

In [ ]:

# ------------------------------------ GADM 1 Emissions ------------------------------------

parquet_path = "zzz_landing_zone/gadm_1_emissions.parquet"
output_path = "gadm_emissions/gadm_1"

# Use DuckDB to write directly from PostgreSQL to Parquet
con = duckdb.connect()


print('Running query')
con.execute(f'''
    INSTALL postgres;
    LOAD postgres;

    CREATE TABLE gadm_1_emissions_parquet AS
    select extract(year from g1e.start_time) as year 
        , g1e.gadm_id
        , gb.gid
        , gb.admin_level
        , g1e.iso3_country
        , ca.name as country_name
        , gb.name gadm_1_name
        , gb.corrected_name gadm_1_corrected_name
        , ca.continent
        , ca.eu
        , ca.oecd
        , ca.unfccc_annex
        , ca.developed_un
        , ca.em_finance
        , ca.g20
        , asch.sector
        , g1e.original_inventory_sector subsector
        , itm.activity_is_temporal
        , g1e.gas
        , case when activity_is_temporal = true then sum(asset_activity) else avg(asset_activity) end as asset_activity
        , sum(asset_emissions) asset_emissions
        , case when activity_is_temporal = true then sum(remainder_activity) else avg(remainder_activity) end as remainder_activity
        , sum(remainder_emissions) remainder_emissions
        , sum(asset_emissions) + sum(remainder_emissions) as emissions_quantity

    from postgres_scan('{postgres_url}', 'public', 'gadm_1_emissions') g1e
    left join (
        select distinct gadm_id
            , gid
            , name
            , corrected_name
            , admin_level
        from postgres_scan('{postgres_url}','public', 'gadm_boundaries') 
        where admin_level = 1
    ) as gb
        on g1e.gadm_id = gb.gadm_id
    left join (
        select distinct sector
            , subsector
        from postgres_scan('{postgres_url}','public', 'asset_schema') 
    ) asch
        on cast(asch.subsector as varchar) = cast(g1e.original_inventory_sector as varchar)
    left join postgres_scan('{postgres_url}','public', 'country_analysis') ca
		on cast(ca.iso3_country as varchar) = cast(g1e.iso3_country as varchar)
    left join postgres_scan('{postgres_url}', 'public', 'is_temporal_map') itm
         on itm.original_inventory_sector = g1e.original_inventory_sector

    where g1e.gas = 'co2e_100yr'
        and extract(year from start_time) = {ers_baseline_year}
        

    group by extract(year from g1e.start_time) 
        , g1e.gadm_id
        , gb.gid
        , gb.admin_level
        , g1e.iso3_country
        , ca.name
        , gb.name 
        , gb.corrected_name
        , ca.continent
        , ca.eu
        , ca.oecd
        , ca.unfccc_annex
        , ca.developed_un
        , ca.em_finance
        , ca.g20
        , asch.sector
        , g1e.original_inventory_sector
        , itm.activity_is_temporal
        , g1e.gas;

    COPY gadm_1_emissions_parquet TO '{parquet_path}' (FORMAT PARQUET);
''')

# and g1e.original_inventory_sector not in ('forest-land-clearing',
#                                                 'forest-land-degradation',
#                                                 'forest-land-fires',
#                                                 'net-forest-land',
#                                                 'net-shrubgrass',
#                                                 'net-wetland',
#                                                 'removals',
#                                                 'shrubgrass-fires',
#                                                 'water-reservoirs',
#                                                 'wetland-fires')
con.close()

split_or_move_parquet(parquet_path, output_path)

print('Successfully refreshed GADM_1 data.')

In [ ]:
# --------------------------------------------------------- GADM 2 BATCH -----------------------------------------------------------------

conn = psycopg2.connect(
    dbname=database,
    user=user,
    password=password,
    host=host,
    port=port
)

cur = conn.cursor(name='parquet_cursor')  # server-side cursor

cur.execute(f"""
     select extract(year from ge.start_time) as year 
        , gb1.gadm_id gadm_1_id
        , gb1.name gadm_1_name
        , gb1.corrected_name gadm_1_corrected_name
        , ge.gadm_id gadm_2_id
        , gb2.name gadm_2_name
        , gb2.corrected_name gadm_2_corrected_name
        , gb2.gid
        , gb2.admin_level
        , ge.iso3_country
        , ca.name as country_name
        , ca.continent
        , ca.eu
        , ca.oecd
        , ca.unfccc_annex
        , ca.developed_un
        , ca.em_finance
        , ca.g20
        , asch.sector
        , ge.original_inventory_sector subsector
        , itm.activity_is_temporal
        , case when activity_is_temporal = true then sum(asset_activity) else avg(asset_activity) end as asset_activity
        , sum(asset_emissions) asset_emissions
        , case when activity_is_temporal = true then sum(remainder_activity) else avg(remainder_activity) end as remainder_activity
        , sum(remainder_emissions) remainder_emissions
        , sum(asset_emissions) + sum(remainder_emissions) as emissions_quantity

    from gadm_emissions ge
    inner join (
        select distinct gadm_id
            , gid
            , immediate_parent
            , name
            , corrected_name
            , admin_level
        from gadm_boundaries
        where admin_level = 2
    ) as gb2
        on ge.gadm_id = gb2.gadm_id
    left join (
        select distinct sector
            , subsector
        from asset_schema
    ) asch
        on cast(asch.subsector as varchar) = cast(ge.original_inventory_sector as varchar)
    left join (
        select gadm_id
            , name
            , corrected_name
        from gadm_boundaries
        where admin_level = 1
    ) gb1
        on gb1.gadm_id = gb2.immediate_parent
    left join country_analysis ca
        on cast(ca.iso3_country as varchar) = cast(ge.iso3_country as varchar)
    left join is_temporal_map itm
        on itm.original_inventory_sector = cast(ge.original_inventory_sector as text)

    where ge.gas = 'co2e_100yr'
        and extract(year from start_time) = {ers_baseline_year}


    group by extract(year from ge.start_time)
        , gb1.gadm_id 
        , gb1.name
        , gb1.corrected_name
        , ge.gadm_id 
        , gb2.name
        , gb2.corrected_name
        , gb2.gid
        , gb2.admin_level
        , ge.iso3_country
        , ca.name
        , ca.continent
        , ca.eu
        , ca.oecd
        , ca.unfccc_annex
        , ca.developed_un
        , ca.em_finance
        , ca.g20
        , asch.sector
        , ge.original_inventory_sector
        , itm.activity_is_temporal
    """)

        # and ge.original_inventory_sector not in ('forest-land-clearing',
        #                                         'forest-land-degradation',
        #                                         'forest-land-fires',
        #                                         'net-forest-land',
        #                                         'net-shrubgrass',
        #                                         'net-wetland',
        #                                         'removals',
        #                                         'shrubgrass-fires',
        #                                         'water-reservoirs',
        #                                         'wetland-fires')

# Set up Parquet writer
batch_size = 10000
output_file = "zzz_landing_zone/gadm_2_emissions.parquet"
output_path = "gadm_emissions/gadm_2"
batch_count = 0
total_rows = 0

print("executing gadm_2 query...")

# Fetch first batch
rows = cur.fetchmany(batch_size)
if not rows:
    raise Exception("No data returned from query.")

field_names = [desc[0] for desc in cur.description]
first_table = pa.Table.from_pylist([dict(zip(field_names, row)) for row in rows])
writer = pq.ParquetWriter(output_file, first_table.schema)
writer.write_table(first_table)
batch_count += 1
total_rows += len(rows)
print(f"Processed batch {batch_count} ({len(rows)} rows), total rows: {total_rows}")

# Process remaining batches
while True:
    rows = cur.fetchmany(batch_size)
    if not rows:
        break

    table = pa.Table.from_pylist([dict(zip(field_names, row)) for row in rows])
    table = table.cast(writer.schema)  # ensure schema matches first batch
    writer.write_table(table)

    batch_count += 1
    total_rows += len(rows)
    print(f"Processed batch {batch_count} ({len(rows)} rows), total rows: {total_rows}")

writer.close()
cur.close()
conn.close()

split_or_move_parquet(output_file, output_path)

print("Successfully refreshed GADM_2 data.")

In [ ]:
# ------------------------------------ City Emissions ------------------------------------

parquet_path = "zzz_landing_zone/city_emissions.parquet"
output_path = "city_emissions"

# Use DuckDB to write directly from PostgreSQL to Parquet
con = duckdb.connect()

print('Running query...')
con.execute( f'''
	INSTALL postgres;
	LOAD postgres;

	CREATE TABLE city_emissions_parquet AS
    
	select extract(year from start_time) as year
		, ce.city_id
		, cb.name as city_name
		, cb.corrected_name as corrected_name
		, ce.iso3_country
		, ca.name as country_name
        , ca.continent
        , ca.eu
        , ca.oecd
        , ca.unfccc_annex
        , ca.developed_un
        , ca.em_finance
        , ca.g20
		, asch.sector
		, ce.original_inventory_sector as subsector
        , itm.activity_is_temporal
		, case when activity_is_temporal = true then sum(asset_activity) else avg(asset_activity) end as asset_activity
		, sum(asset_emissions) asset_emissions
		, case when activity_is_temporal = true then sum(remainder_activity) else avg(remainder_activity) end as remainder_activity
		, sum(remainder_emissions) remainder_emissions
		, sum(asset_emissions) + sum(remainder_emissions) as emissions_quantity

	from postgres_scan('{postgres_url}','public', 'city_emissions') ce
	left join postgres_scan('{postgres_url}','public', 'city_boundaries') cb
		on cb.city_id = ce.city_id
        and cb.reporting_entity = 'ghs-fua'
	left join (
		select distinct sector, subsector
		from postgres_scan('{postgres_url}','public', 'asset_schema')
	) asch
		on cast(asch.subsector as varchar) = cast(ce.original_inventory_sector as varchar)
	left join postgres_scan('{postgres_url}','public', 'country_analysis') ca
		on cast(ca.iso3_country as varchar) = cast(ce.iso3_country as varchar)
    left join postgres_scan('{postgres_url}', 'public', 'is_temporal_map') itm
         on itm.original_inventory_sector = ce.original_inventory_sector

	where extract(year from ce.start_time) = {ers_baseline_year}
		and ce.gas = 'co2e_100yr'
        and cb.city_id is not null

	group by extract(year from start_time) 
		, ce.city_id
		, cb.name 
		, cb.corrected_name 
		, ce.iso3_country
		, ca.name 
        , ca.continent
        , ca.eu
        , ca.oecd
        , ca.unfccc_annex
        , ca.developed_un
        , ca.em_finance
        , ca.g20
		, asch.sector
		, ce.original_inventory_sector
        , itm.activity_is_temporal;
            
    COPY city_emissions_parquet TO '{parquet_path}' (FORMAT PARQUET);
            
    ''')

# and ce.original_inventory_sector not in ('forest-land-clearing',
# 														'forest-land-degradation',
# 														'forest-land-fires',
# 														'net-forest-land',
# 														'net-shrubgrass',
# 														'net-wetland',
# 														'removals',
# 														'shrubgrass-fires',
# 														'water-reservoirs',
# 														'wetland-fires')

con.close()

split_or_move_parquet(parquet_path, output_path)

print('Successfuly refreshed city_emissions data.')

In [ ]:
# ------------------------------------ Asset Ownership ------------------------------------

parquet_path = "zzz_landing_zone/asset_ownership.parquet"
output_path = "ownership"

# Use DuckDB to write directly from PostgreSQL to Parquet
con = duckdb.connect()

print('Running query...')
con.execute( f'''
	INSTALL postgres;
	LOAD postgres;

	CREATE TABLE asset_ownership_parquet AS
            
    SELECT *
    FROM postgres_scan('{postgres_url}','public', 'asset_ownership');
    
    COPY asset_ownership_parquet TO '{parquet_path}' (FORMAT PARQUET);
            
    ''')

con.close()

split_or_move_parquet(parquet_path, output_path)

print('Successfully refreshed ownership data.')

In [ ]:
# ------------------------------------ Demographic ------------------------------------

parquet_path = "zzz_landing_zone/demographic.parquet"
output_path = "demographic"

con = duckdb.connect()

print('Running query...')
con.execute( f'''
	INSTALL postgres;
	LOAD postgres;

	CREATE TABLE demographic_parquet AS
            
    select *
    from postgres_scan('{postgres_url}', 'public', 'demographic_data')
    where version = 'global_pop_2024_CN_1km_R2024B_UA_v1';
    
    COPY demographic_parquet TO '{parquet_path}' (FORMAT PARQUET);
            
    ''')

con.close()

split_or_move_parquet(parquet_path, output_path)

print('Successfully refreshed ownership data.')

In [ ]:
# ------------------------------------ Country-Region mapping ------------------------------------

parquet_path = "zzz_landing_zone/country_region_mapping.parquet"
output_path = "country_region_mapping"

con = duckdb.connect()

print('Running query...')
con.execute( f'''
	INSTALL postgres;
	LOAD postgres;

	CREATE TABLE country_region_mapping AS
            
    select distinct iso3_country
        , name as country_name
        , continent
        , region 
    from postgres_scan('{postgres_url}', 'public', 'country_analysis')
    ;
    
    COPY country_region_mapping TO '{parquet_path}' (FORMAT PARQUET);
            
    ''')

con.close()

split_or_move_parquet(parquet_path, output_path)

print('Successfully refreshed country_mappings data.')

In [ ]:
print('Data refresh complete!')